# CRP-DPGMM five-seeds

This notebook is based on `CRP-DPGMM.ipynb`. It runs CRP-DPGMM five times with different seeds and reports mean and sample standard deviation for ACC, DDA, Brier score, ECE, and runtime.


In [1]:
import numpy as np
import matplotlib.pyplot as plt
from CRP_DPGMM import CRP_DpMixture

import torch
from scipy.optimize import linear_sum_assignment
from typing import List, Callable, Union, Any, TypeVar, Tuple
Tensor = TypeVar('torch.tensor')
import pandas as pd
from sklearn.decomposition import PCA
import time
from Uncertainty_calibration_scores import *
seed = 42
torch.manual_seed(seed)  # ensure reproducible results
np.random.seed(seed)
from collections import Counter


# Dataset


In [ ]:
'''use a numerical dataset to test the model
label  damaged_floor     damage_extent         number_of_samples
0           0            0%                     1500
1           1            3%                     500
2           1            6%                     500
3           1            10%                    500
4           1,3          5%,10%                 500
5           1,3,5        5%,10%,15%             500
6           2,4,6        10%,15%,20%            500
7           1,3,5,7      10%,15%,20%,25%        500


'''
# Read data from CSV
features = pd.read_csv("TF_mag_numerical_8class_5000samples_20dB.csv")
features = features.astype("float32")
# # # Convert DataFrame to PyTorch tensors
X = torch.tensor(features.values[:7500,:])
X1 = X[1000:1500,:]
X2 = X[1800:2300,:]
X = torch.cat([X1, X2], dim=0)

X = X.t()

input_dim = X.shape[1]
print(X.shape)


a = []
for i in range(1500):
    a.append(0)
for i in np.arange(1500,2000):
    a.append(1)
for i in np.arange(2000,2500):
    a.append(2)
for i in np.arange(2500,3000):
    a.append(3)
for i in np.arange(3000,3500):
    a.append(4)
for i in np.arange(3500,4000):
    a.append(5)
for i in np.arange(4000,4500):
    a.append(6)
for i in np.arange(4500,5000):
    a.append(7)
print(len(a))

y0 = torch.tensor(a)
y0 = y0.unsqueeze(1)
y = [int(_) for _ in y0] 
y = torch.tensor(y)
num_classes = y.max().item() + 1
print(f"number of total classes: {num_classes}")

plt.plot(y)
plt.show()

# PCA data for CRP-DPGMM


In [ ]:
pca_components = 10
samples_per_segment = 100

pca = PCA(n_components=pca_components)
data_pca = pca.fit_transform(X).astype("float32")
data_pca = data_pca.T
data_pca = torch.tensor(data_pca)
print('PCA data shape:', data_pca.shape)
data_gibbs = data_pca.numpy().T
print('data for gibbs shape:', data_gibbs.shape)

# Five-seed CRP-DPGMM helpers


In [4]:
def unsupervised_clustering_accuracy(y: Union[np.ndarray, torch.Tensor], y_pred: Union[np.ndarray, torch.Tensor]) -> tuple:
    """Unsupervised Clustering Accuracy
    """
    assert len(y_pred) == len(y)
    u = np.unique(y)
    n_true_clusters = len(u)
    v = np.unique(y_pred)
    n_pred_clusters = len(v)
    map_u = dict(zip(u, range(n_true_clusters)))
    map_v = dict(zip(v, range(n_pred_clusters)))
    inv_map_u = {v: k for k, v in map_u.items()}
    inv_map_v = {v: k for k, v in map_v.items()}
    r = np.zeros((n_pred_clusters, n_true_clusters), dtype=np.int64)
    for y_pred_, y_ in zip(y_pred, y):
        if y_ in map_u:
            r[map_v[y_pred_], map_u[y_]] += 1
    reward_matrix  = np.concatenate((r, r, r), axis=1)
    cost_matrix = reward_matrix.max() - reward_matrix
    row_assign, col_assign = linear_sum_assignment(cost_matrix)

    # Construct optimal assignments matrix
    row_assign = row_assign.reshape((-1, 1))  # (n,) to (n, 1) reshape
    col_assign = col_assign.reshape((-1, 1))  # (n,) to (n, 1) reshape
    assignments = np.concatenate((row_assign, col_assign), axis=1)
    assignments = [[inv_map_v[x], inv_map_u[y%n_true_clusters]] for x, y in assignments]

    optimal_reward = reward_matrix[row_assign, col_assign].sum() * 1.0
    return optimal_reward / y_pred.size, assignments  

def damage_detection_accuracy(assignments, healthy_count=1500, healthy_reference_fraction=0.8, min_count=10):
    assignments = np.asarray(assignments).astype(int)
    healthy_reference_end = int(healthy_count * healthy_reference_fraction)
    health_labels_pre = np.unique(assignments[:healthy_reference_end])
    counts = Counter(assignments[:healthy_reference_end])
    health_labels = [label for label in health_labels_pre if counts[label] >= min_count]

    fn, fp = 0, 0
    for i in range(len(assignments)):
        if i <= healthy_count and assignments[i] not in health_labels:
            fp += 1
        elif i > healthy_count and assignments[i] in health_labels:
            fn += 1

    dda = 1 - (fp + fn) / len(assignments)
    return dda, fp, fn, health_labels

def build_crp_components(DP):
    component_id = 0
    CRP_comps = {}
    CRP_nk = []

    for comp in DP.params.values():
        if comp.nk <= 0:
            continue

        _kappa_0 = comp._kappa_0
        _nu_0 = comp._nu_0
        _mu_0 = comp._mu_0
        _Psi_0 = comp._Psi_0
        nk = comp.nk
        x_bar_k = comp.ss['x_bar_k']
        _square_sum = comp._square_sum

        kappa_k = _kappa_0 + nk
        nu_k = _nu_0 + nk
        x_bar_k_mu_0 = x_bar_k - _mu_0
        S_k = _square_sum - nk * (x_bar_k.transpose() * x_bar_k)
        Psi_k = _Psi_0 + S_k + _kappa_0 * nk * x_bar_k_mu_0.transpose() * x_bar_k_mu_0 / (_kappa_0 + nk)
        mu_k = (_kappa_0 * _mu_0 + nk * x_bar_k) / (_kappa_0 + nk)

        CRP_comps[component_id] = {
            'kappa_n': kappa_k,
            'nu_n': nu_k,
            'mu_n': mu_k,
            'Psi_n': Psi_k,
        }
        CRP_nk.append(nk)
        component_id += 1

    return CRP_comps, CRP_nk

def run_crp_dpgmm_once(seed, X, y_tensor, y_true, iterations=200, sample_alpha=True, alpha_a=1.0, alpha_b=1.0):
    np.random.seed(seed)
    torch.manual_seed(seed)

    pca_components = 10
    pca = PCA(n_components=pca_components)
    data_pca_seed = pca.fit_transform(X).astype("float32")
    data_pca_seed = data_pca_seed.T
    data_pca_seed = torch.tensor(data_pca_seed)
    data = data_pca_seed.numpy().T

    dim = data.shape[1]
    alpha = 1.0
    hyperparameter = {
        'mu_0': np.zeros((1, dim)),
        'kappa_0': 1.0,
        'nu_0': 2.0,
        'Psi_0': np.eye(dim),
    }

    DP = CRP_DpMixture.DpMixture(
        data,
        hyperparameter,
        alpha,
        sample_alpha=sample_alpha,
        alpha_a=alpha_a,
        alpha_b=alpha_b,
    )

    start_time = time.time()
    DP.gibbs(iterations=iterations, snapshot_interval=1, y=y_tensor)
    runtime = time.time() - start_time

    pred = DP.assigns.astype(int)
    acc, assignments = unsupervised_clustering_accuracy(y_true, pred)
    dda, fp, fn, health_labels = damage_detection_accuracy(pred)

    CRP_comps, CRP_nk = build_crp_components(DP)
    resp = dpgmm_crp_responsibilities_closed(
        X=data,
        components=CRP_comps,
        n_k=CRP_nk,
        eps=1e-300,
    )

    BS, BSc, BSS, P_hat, R, Q, classes = dpgmm_brier_from_responsibilities(
        R=resp,
        y_true=y_true,
        labeled_mask=None,
        classes=None,
        sample_weight=None,
    )

    P_dpg, _ = probs_with_cv_Q(R=resp, y_true=y_true, classes=classes, kfold=5)
    ece_dpg_top = ece_toplabel(P_dpg, y_true, n_bins=15)
    ece_dpg_ovr = ece_ovr(P_dpg, y_true=y_true, n_bins=15, classes=classes)
    ece_dpg_balanced = ece_ovr_classbalanced(P_dpg, y_true, n_bins=15)

    return {
        'seed': seed,
        'accuracy': acc,
        'damage_detection_accuracy': dda,
        'brier_score': BS,
        'brier_skill_score': BSS,
        'ece_toplabel': ece_dpg_top,
        'ece_ovr': ece_dpg_ovr,
        'ece_balanced': ece_dpg_balanced,
        'runtime_seconds': runtime,
        'num_components': DP._K,
        'alpha': DP.alpha,
        'false_positive': fp,
        'false_negative': fn,
        'health_labels': health_labels,
        'assignments': assignments,
    }

# Experiment configuration


In [ ]:
experiment_seeds = [42, 1, 2, 3, 6]
iterations = 200

# data = data_gibbs
y_true = y.numpy().astype(int)

# Run five-seed CRP-DPGMM experiment


In [ ]:
seed_results = []

for run_id, seed in enumerate(experiment_seeds, start=1):
    print(f"\n===== Seed {seed} ({run_id}/{len(experiment_seeds)}) =====")
    result = run_crp_dpgmm_once(
        seed=seed,
        X=X,
        y_tensor=y,
        y_true=y_true,
        iterations=iterations,
        sample_alpha=True,
    )
    seed_results.append(result)
    print(
        f"Seed {seed}: ACC={result['accuracy']:.6f}, "
        f"DDA={result['damage_detection_accuracy']:.6f}, "
        f"Brier={result['brier_score']:.6f}, "
        f"ECE_top={result['ece_toplabel']:.6f}, "
        f"runtime={result['runtime_seconds']:.2f}s, "
        f"K={result['num_components']}"
    )

results_df = pd.DataFrame(seed_results)
metric_columns = [
    'accuracy',
    'damage_detection_accuracy',
    'brier_score',
    'ece_balanced',
    'runtime_seconds',
]
summary_df = pd.DataFrame({
    'mean': results_df[metric_columns].mean(),
    'std': results_df[metric_columns].std(ddof=1),
})

print("\nPer-seed results")
display(results_df[[
    'seed',
    'accuracy',
    'damage_detection_accuracy',
    'brier_score',
    'ece_balanced',
    'runtime_seconds',
    'num_components',
    'alpha',
]])

print("\nMean and sample standard deviation over seeds")
display(summary_df)
